# 🦷 Dental Caries Detection & Segmentation using UNet++ (Colab GPU Training)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mohitinnani004-afk/dental-caries-detection-unetpp/blob/main/notebooks/dental_caries_colab_training.ipynb)

This complete notebook walks through:
1. **Environment Setup & GPU Check** on Google Colab.
2. **Detailed Step-by-Step Mask Creation** from raw radiographs & polygon coordinates without black-box functions.
3. **Radiograph Preprocessing** using Contrast Limited Adaptive Histogram Equalization (CLAHE).
4. **UNet++ (Nested U-Net) Model Training** with Deep Supervision ($L_1-L_4$) & Spatial Attention Gates.
5. **Quantitative & Lesion-Level Evaluation** with ICDAS severity classification.
6. **Exporting Trained Checkpoints** to Google Drive or local download.

## 🛠️ Step 1: Verify GPU & Clone Repository

In [ ]:
# Check CUDA GPU availability in Colab (Go to Runtime -> Change runtime type -> T4 GPU if not enabled)
!nvidia-smi

import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")

In [ ]:
# Clone the GitHub repository and enter the directory
!git clone https://github.com/mohitinnani004-afk/dental-caries-detection-unetpp.git
%cd dental-caries-detection-unetpp

# Install dependencies
!pip install -q -r requirements.txt

## 🔬 Step 2: Detailed Mask Creation & Inspection (No Black-Box Functions)

Dental caries lesions are radiolucent (dark) demineralizations in tooth enamel or dentin. In clinical machine learning, ground-truth masks must be transparently created from annotation coordinate vertices $[(x_1, y_1), (x_2, y_2), \dots]$.

Below, we explicitly rasterize coordinate polygons into a 2D binary matrix `[H, W]` and generate a **detailed 5-panel inspection view**, including a **zoomed-in pixel grid** showing exact caries boundary pixels.

In [ ]:
import numpy as np
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt
import matplotlib.patches as patches

def create_binary_mask_explicit(image_shape, polygon_coords):
    """
    Explicitly rasterizes a list of (x, y) boundary coordinates into a binary matrix [H, W].
    Pixel = 1 if inside caries lesion polygon, else 0 (healthy tooth/bone).
    """
    h, w = image_shape
    # 1. Start with blank matrix of zeros (Sound tooth structure)
    mask = Image.new("L", (w, h), 0)
    
    # 2. Fill the polygon defined by coordinates
    draw = ImageDraw.Draw(mask)
    draw.polygon(polygon_coords, fill=1)
    
    # 3. Convert to numpy matrix
    mask_np = np.array(mask, dtype=np.uint8)
    return mask_np

# Load a sample dental radiograph
raw_img = Image.open("samples/sample_proximal.png").convert("L")
img_array = np.array(raw_img)
H, W = img_array.shape

# Define clinical polygon coordinates for an interproximal molar caries lesion
lesion_polygon = [
    (108, 92), (116, 84), (124, 88), (128, 98),
    (126, 110), (118, 116), (110, 112), (105, 102)
]

# Create the binary mask matrix
binary_mask = create_binary_mask_explicit((H, W), lesion_polygon)

print(f"Mask Matrix Shape: {binary_mask.shape}")
print(f"Unique Matrix Values: {np.unique(binary_mask)} (0 = Sound, 1 = Caries)")
print(f"Total Caries Pixels: {np.sum(binary_mask):,} px")
print(f"Estimated Lesion Area: {np.sum(binary_mask) * (0.08**2):.2f} mm² (at 0.08 mm/pixel pitch)")

In [ ]:
# Render 5-Panel Detailed Walkthrough Figure
fig, axes = plt.subplots(1, 5, figsize=(24, 5.2), facecolor="#0e1117")
plt.subplots_adjust(wspace=0.15)

for ax in axes:
    ax.set_facecolor("#0e1117")
    ax.tick_params(colors="#8892b0")
    for spine in ax.spines.values():
        spine.set_color("#233554")

# Panel 1: Raw Radiograph
axes[0].imshow(img_array, cmap="gray")
axes[0].set_title("1. Raw Dental Radiograph", color="white", fontsize=11, fontweight="bold")
axes[0].set_xlabel(f"{W} px", color="#8892b0")
axes[0].set_ylabel(f"{H} px", color="#8892b0")

# Panel 2: Polygon Boundary Coordinates
axes[1].imshow(img_array, cmap="gray")
poly = patches.Polygon(lesion_polygon, closed=True, fill=False, edgecolor="#ff4757", linewidth=2.5)
axes[1].add_patch(poly)
vx, vy = zip(*lesion_polygon)
axes[1].scatter(vx, vy, color="#00f2fe", s=35, zorder=5)
axes[1].text(116, 75, "Lesion #1\n(Dentin D1)", color="#ffa502", fontsize=9, fontweight="bold",
             bbox=dict(boxstyle="round,pad=0.2", facecolor="#1a1c23", edgecolor="#ffa502"))
axes[1].set_title("2. Polygon Annotations", color="white", fontsize=11, fontweight="bold")

# Panel 3: Binary Mask Matrix
axes[2].imshow(binary_mask * 255, cmap="gray")
axes[2].set_title(f"3. Rasterized Mask ({np.sum(binary_mask)} px)", color="white", fontsize=11, fontweight="bold")

# Panel 4: Alpha-blended Overlay
overlay = np.stack([img_array]*3, axis=-1).astype(np.float32) / 255.0
mask_idx = binary_mask > 0
overlay[mask_idx] = overlay[mask_idx] * 0.35 + np.array([1.0, 0.28, 0.34]) * 0.65
axes[3].imshow(overlay)
axes[3].set_title("4. Clinical Diagnostic Overlay", color="white", fontsize=11, fontweight="bold")

# Panel 5: Zoomed Inset with Pixel Grid
cx, cy = 116, 100
pad = 22
crop_overlay = overlay[cy-pad:cy+pad, cx-pad:cx+pad]
axes[4].imshow(crop_overlay, interpolation="nearest")
axes[4].set_xticks(np.arange(-0.5, pad*2, 1), minor=True)
axes[4].set_yticks(np.arange(-0.5, pad*2, 1), minor=True)
axes[4].grid(which="minor", color="#00f2fe", linestyle="-", linewidth=0.5, alpha=0.35)
axes[4].set_title(f"5. Zoomed Pixel Grid ({pad*2}x{pad*2} px)", color="white", fontsize=11, fontweight="bold")

plt.tight_layout()
plt.show()

## 📊 Step 3: Dataset Preparation & Preprocessing (CLAHE)

In [ ]:
from data.sample_generator import generate_sample_dataset
from data.dataset import create_dental_dataloaders
from pathlib import Path

# Prepare or bootstrap dataset with 64 radiograph/mask pairs
dataset_dir = Path("data/dataset")
print("Bootstrapping synthetic dental radiograph dataset for training...")
generate_sample_dataset(output_dir=dataset_dir, num_samples=64, width=256, height=256)

# Create PyTorch DataLoaders (80% Train, 10% Val, 10% Test) with CLAHE preprocessing
train_loader, val_loader, test_loader = create_dental_dataloaders(
    data_dir=dataset_dir,
    image_size=(256, 256),
    batch_size=8,
    train_split=0.8,
    val_split=0.1,
    test_split=0.1,
    apply_clahe=True
)

print(f"\nDataLoaders Initialized:")
print(f"  Training batches:   {len(train_loader)} ({len(train_loader.dataset)} samples)")
print(f"  Validation batches: {len(val_loader)} ({len(val_loader.dataset)} samples)")
print(f"  Test batches:       {len(test_loader)} ({len(test_loader.dataset)} samples)")

## 🧠 Step 4: Instantiate UNet++ Architecture

We instantiate **UNet++ (Nested U-Net)** with:
- Dense skip pathways ($X^{i,j}$)
- Additive Spatial Attention Gates on skip connections
- Deep Supervision heads at $X^{0,1}, X^{0,2}, X^{0,3}, X^{0,4}$

In [ ]:
from models.unet_plus_plus import build_unet_plus_plus

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Target compute device: {device}")

model = build_unet_plus_plus(
    in_channels=1,
    num_classes=1,
    deep_supervision=True,
    use_attention=True,
    nb_filter=[32, 64, 128, 256, 512]
).to(device)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"UNet++ Architecture successfully compiled!")
print(f"Total trainable parameters: {total_params:,}")

## 🎯 Step 5: Loss Function, Optimizer & Learning Rate Schedule

In [ ]:
from models.losses import BCEDiceLoss, DeepSupervisionLoss
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

NUM_EPOCHS = 35

# Hybrid BCE + Dice loss with deep supervision weighting
base_loss = BCEDiceLoss(bce_weight=0.5, dice_weight=0.5)
criterion = DeepSupervisionLoss(base_criterion=base_loss, weights=[0.1, 0.2, 0.3, 0.4])

# AdamW optimizer with Cosine Annealing decay
optimizer = AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS, eta_min=1e-6)

print("Loss, Optimizer & Scheduler successfully configured.")

## 🚀 Step 6: Train UNet++ on GPU

In [ ]:
from training.trainer import Trainer

trainer = Trainer(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    criterion=criterion,
    scheduler=scheduler,
    device=device,
    checkpoint_dir="checkpoints",
    deep_supervision=True,
    mixed_precision=torch.cuda.is_available(),
    early_stopping_patience=10
)

# Start training
history = trainer.fit(num_epochs=NUM_EPOCHS)

## 📈 Step 7: Plot Training & Validation Curves

In [ ]:
epochs_range = range(1, len(history["train_loss"]) + 1)

plt.figure(figsize=(16, 5), facecolor="#0e1117")

# Loss plot
plt.subplot(1, 3, 1, facecolor="#0e1117")
plt.plot(epochs_range, history["train_loss"], color="#ff4757", label="Train Loss", linewidth=2)
plt.plot(epochs_range, history["val_loss"], color="#ffa502", linestyle="--", label="Val Loss", linewidth=2)
plt.title("Loss Curve (BCE + Dice)", color="white", fontweight="bold")
plt.xlabel("Epoch", color="#8892b0")
plt.ylabel("Loss", color="#8892b0")
plt.tick_params(colors="#8892b0")
plt.legend()
plt.grid(alpha=0.15)

# Dice Similarity plot
plt.subplot(1, 3, 2, facecolor="#0e1117")
plt.plot(epochs_range, history["val_dice"], color="#00f2fe", label="Val Dice (DSC)", linewidth=2)
plt.title("Validation Dice Similarity", color="white", fontweight="bold")
plt.xlabel("Epoch", color="#8892b0")
plt.ylabel("Dice Score", color="#8892b0")
plt.tick_params(colors="#8892b0")
plt.legend()
plt.grid(alpha=0.15)

# IoU plot
plt.subplot(1, 3, 3, facecolor="#0e1117")
plt.plot(epochs_range, history["val_iou"], color="#10b981", label="Val IoU (Jaccard)", linewidth=2)
plt.title("Validation IoU (Jaccard Index)", color="white", fontweight="bold")
plt.xlabel("Epoch", color="#8892b0")
plt.ylabel("IoU Score", color="#8892b0")
plt.tick_params(colors="#8892b0")
plt.legend()
plt.grid(alpha=0.15)

plt.tight_layout()
plt.show()

## 🩺 Step 8: Quantitative Test Set Evaluation & Diagnostic Plates

In [ ]:
from evaluation.evaluator import Evaluator

# Evaluate best saved checkpoint
evaluator = Evaluator(model=model, device=device, threshold=0.50)
eval_results = evaluator.evaluate(test_loader=test_loader, save_visualizations=True, output_dir="outputs/colab_eval")

pix = eval_results["pixel_metrics"]
les = eval_results.get("lesion_metrics", {})

print("=" * 50)
print("           TEST SET EVALUATION RESULTS")
print("=" * 50)
print(f"  Dice Similarity (DSC): {pix.get('dice', 0.0):.4f}")
print(f"  Intersection over Union: {pix.get('iou', 0.0):.4f}")
print(f"  Pixel Sensitivity:     {pix.get('sensitivity', 0.0):.4f}")
print(f"  Pixel Specificity:     {pix.get('specificity', 0.0):.4f}")
print(f"  Pixel Precision:       {pix.get('precision', 0.0):.4f}")
if les:
    print("-" * 50)
    print(f"  Lesion Detection F1:   {les.get('lesion_f1', 0.0):.4f}")
    print(f"  Lesion Recall:         {les.get('lesion_recall', 0.0):.4f}")
    print(f"  Lesion Precision:      {les.get('lesion_precision', 0.0):.4f}")
print("=" * 50)

## 🔍 Step 9: Clinical Lesion Extraction & ICDAS Severity Grading

In [ ]:
from inference.predictor import CariesPredictor

predictor = CariesPredictor(
    checkpoint_path="checkpoints/best_model.pth",
    device=device,
    threshold=0.50
)

# Run inference on demonstration sample
result = predictor.predict("samples/sample_proximal.png")

print(f"Diagnostic Summary:")
print(f"  Caries Identified: {result['summary']['caries_detected']}")
print(f"  Number of Lesions: {result['summary']['lesion_count']}")
print(f"  Highest Severity:  {result['summary']['highest_severity']}")
print(f"  Total Area:        {result['summary']['estimated_caries_area_mm2']} mm²\n")

print("Individual Lesion Findings:")
for lesion in result["lesions"]:
    print(f"  * Lesion #{lesion['id']} [{lesion['site']}]: {lesion['severity_grade']} (Conf: {int(lesion['confidence']*100)}%)")
    print(f"    Bounding Box: {lesion['bbox']}")
    print(f"    Clinical Recommendation: {lesion['clinical_recommendation']}\n")

# Display annotated visualization
result["annotated_image"].resize((450, 450))

## 💾 Step 10: Download Checkpoint to Local Machine or Google Drive

In [ ]:
# Download the trained model checkpoint directly to your local computer
try:
    from google.colab import files
    files.download("checkpoints/best_model.pth")
    print("Downloaded checkpoints/best_model.pth to your computer!")
except Exception as e:
    print(f"Checkpoint saved at: checkpoints/best_model.pth ({e})")